# PV & BESS Hosting Capacity — IEEE 33-Bus MILP
### Improved Accuracy Version

**Key improvements over original notebook:**
1. **Full LinDistFlow (P+Q)** — reactive power and X terms included; voltage drops now accurate for inductive lines.
2. **SOC bounds linearised** — `E[n,t] ∈ [0, Emax[n]]` replaces the bilinear `E >= SOC_min·Emax` form (Var×Var is not LP/MILP-safe in HiGHS). `Emax[n]` is now the **usable** energy window; physical rated capacity = Emax / 0.70.
3. **Simultaneous charge/discharge prevented** — binary `u_ch[n,t]` mutex added (original allowed both at once, inflating apparent capacity).
4. **Reactive load modelled** — Q injections from PV (unity PF → Q_pv = 0) and reactive balance per bus; Q-flow variables added.
5. **Voltage-drop uses both R and X** — `V_j = V_i − R·P − X·Q`; purely resistive drop under-estimates voltage sag on inductive lines (branches 6,7,12,16,19,21,23 have X > R).
6. **Hosting capacity metric added** — `HC_PV = Σ PPVmax_kW` and `HC_BESS_E = Σ Emax_kWh` printed explicitly.
7. **Cycle constraint corrected** — SOC at end of day equals *initial* SOC (not hard-coded 0), so any feasible starting SOC is accepted.
8. **PV curtailment variable added** — distinguishes available PV from dispatched PV for accurate hosting capacity.


## 1. Data — IEEE 33-Bus, Load & PV Profiles

In [ ]:
import numpy as np
import pandas as pd
from pyomo.environ import (
    ConcreteModel, Set, Param, Var, NonNegativeReals, Binary, Reals,
    Constraint, Objective, minimize, value
)
from pyomo.opt import SolverFactory, TerminationCondition

# ══════════════════════════════════════════════════════════════════
#  BASE VALUES  —  IEEE 33-bus (Baran-Wu 1989)
#  Vbase = 12.66 kV (L-L),  Sbase = 10 MVA  →  Zbase = 16.03 Ω
#  The branch R,X values in this dataset are in OHMS.
#  We normalise them to pu here so that LinDistFlow is dimensionally
#  consistent:   V_j = V_i  −  R_pu · P_pu  −  X_pu · Q_pu
# ══════════════════════════════════════════════════════════════════
Vbase_kV   = 12.66          # kV  (line-to-line)
Sbase_kW   = 10_000.0       # kW  (= 10 MVA)
Sbase_kVAr = 10_000.0
Zbase_ohm  = (Vbase_kV**2 * 1e6) / (Sbase_kW * 1e3)   # = 16.03 Ω

print(f"Zbase = {Zbase_ohm:.4f} Ω   (Vbase={Vbase_kV} kV, Sbase={Sbase_kW} kW)")

nb   = 33
slack = 1

# ─── Bus loads (kW, kVAr) ────────────────────────────────────────
bus_load = {
    1: (0,0),     2: (100,60),   3: (90,40),   4: (120,80),
    5: (60,30),   6: (60,20),    7: (200,100),  8: (200,100),
    9: (60,20),  10: (60,20),   11: (45,30),   12: (60,35),
   13: (60,35),  14: (120,80),  15: (60,10),   16: (60,20),
   17: (60,20),  18: (90,40),   19: (90,40),   20: (90,40),
   21: (90,40),  22: (90,40),   23: (90,50),   24: (420,200),
   25: (420,200),26: (60,25),   27: (60,25),   28: (60,20),
   29: (120,70), 30: (200,600), 31: (150,70),  32: (210,100),
   33: (60,40),
}

# ─── Branches: (from, to, R_ohm, X_ohm) — converted to pu below ─
branches_raw = {
     1:(1, 2, 0.0922,0.0470),   2:(2, 3, 0.4930,0.2511),
     3:(3, 4, 0.3660,0.1864),   4:(4, 5, 0.3811,0.1941),
     5:(5, 6, 0.8190,0.7070),   6:(6, 7, 0.1872,0.6188),
     7:(7, 8, 1.7114,1.2351),   8:(8, 9, 1.0300,0.7400),
     9:(9,10, 1.0440,0.7400),  10:(10,11,0.1966,0.0650),
    11:(11,12,0.3744,0.1238),  12:(12,13,1.4680,1.1550),
    13:(13,14,0.5416,0.7129),  14:(14,15,0.5910,0.5260),
    15:(15,16,0.7463,0.5450),  16:(16,17,1.2890,1.7210),
    17:(17,18,0.7320,0.5740),  18:(2, 19,0.1640,0.1565),
    19:(19,20,1.5042,1.3554),  20:(20,21,0.4095,0.4784),
    21:(21,22,0.7089,0.9373),  22:(3, 23,0.4512,0.3083),
    23:(23,24,0.8980,0.7091),  24:(24,25,0.8960,0.7011),
    25:(6, 26,0.2030,0.1034),  26:(26,27,0.2842,0.1447),
    27:(27,28,1.0590,0.9337),  28:(28,29,0.8042,0.7006),
    29:(29,30,0.5075,0.2585),  30:(30,31,0.9744,0.9630),
    31:(31,32,0.3105,0.3619),  32:(32,33,0.3410,0.5302),
}

# Convert to per-unit
branches = {
    l: (fi, ti, r/Zbase_ohm, x/Zbase_ohm)
    for l, (fi, ti, r, x) in branches_raw.items()
}

# ─── 24-hour profiles ────────────────────────────────────────────
T = list(range(24))
load_mult = np.array([
    0.685,0.659,0.644,0.644,0.674,0.731,
    0.768,0.830,0.882,0.916,0.959,0.989,
    0.987,0.974,0.968,0.965,0.950,0.929,
    0.990,1.000,0.958,0.887,0.804,0.735,
], dtype=float)
pv_cf = np.array([
    0,0,0,0,0,0,
    0.10090,0.32375,0.56753,0.75722,0.91625,0.98658,
    0.75912,0.73428,0.56403,0.41945,0.21907,0.05410,
    0,0,0,0,0,0,
], dtype=float)
dt = 1.0  # hours

# ─── Adjacency (precomputed — never use value() inside constraints) ──
from_bus  = {l: branches[l][0] for l in branches}
to_bus    = {l: branches[l][1] for l in branches}
in_lines  = {n: [l for l in branches if to_bus[l]  == n] for n in range(1, nb+1)}
out_lines = {n: [l for l in branches if from_bus[l] == n] for n in range(1, nb+1)}

# ─── Quick sanity check: base-case min voltage ───────────────────
from collections import deque
_ch={n:[] for n in range(1,nb+1)}; _pb={}
for l,(fi,ti,r,x) in branches.items(): _ch[fi].append(ti); _pb[ti]=l
_bfs=[1]; _q=deque([1])
while _q:
    _node=_q.popleft()
    for c in _ch[_node]: _bfs.append(c); _q.append(c)
_P={l:0.0 for l in branches}; _Q={l:0.0 for l in branches}
for _node in reversed(_bfs[1:]):
    l=_pb[_node]; fi,ti,r,x=branches[l]
    _out=[ll for ll,(fii,tii,*_) in branches.items() if fii==_node]
    _P[l]=bus_load[_node][0]/Sbase_kW+sum(_P[ll] for ll in _out)
    _Q[l]=bus_load[_node][1]/Sbase_kVAr+sum(_Q[ll] for ll in _out)
_V={1:1.0}
for _node in _bfs[1:]:
    l=_pb[_node]; fi,ti,r,x=branches[l]
    _V[_node]=_V[fi]-r*_P[l]-x*_Q[l]
_minV=min(_V.values()); _minbus=min(_V,key=_V.get)
print(f"Base-case min voltage (no DER, peak load): {_minV:.4f} pu  at bus {_minbus}")
print(f"  Literature reference (Baran-Wu 1989): ~0.9131 pu at bus 18")
print(f"  Max Q-flow: {max(_Q.values()):.4f} pu = {max(_Q.values())*Sbase_kVAr:.0f} kVAr")
print("Data loaded OK.")


## 2. MILP Formulation — Full LinDistFlow (P + Q)

**Voltage-drop equation (per branch l, hour t):**

$$V_j(t) = V_i(t) - R_l \cdot P_l(t) - X_l \cdot Q_l(t)$$

This replaces the P-only approximation `V_j = V_i − R·P` in the original,
which underestimates voltage sag on branches with significant reactance (e.g., branches 6, 7, 12, 16).

**SOC dynamics (fix #2 & #3):**

$$E_{n,t} = E_{n,t-1} + \eta_c P^{ch}_{n,t} \Delta t - \frac{P^{dis}_{n,t}}{\eta_d} \Delta t$$

$$0.20 \cdot E^{max}_n \le E_{n,t} \le 0.90 \cdot E^{max}_n$$

$$u^{ch}_{n,t} + u^{dis}_{n,t} \le 1 \quad \text{(mutex)}$$


In [ ]:
# ─── Planning limits ──────────────────────────
Npv_max   = 5
Nbess_max = 3

PV_cap_max_kW  = 5000.0    # per-site max (kW)
BESS_p_max_kW  = 2000.0    # per-site charge/discharge max (kW)
BESS_e_max_kWh = 5000.0    # per-site energy capacity max (kWh)

Vmin = 0.90;  Vmax = 1.05;  Vref = 1.0  # 0.90 pu: base network min is 0.9115 pu (Baran-Wu)
Pline_max_pu = 1.5   # pu thermal limit (≈15 MW for Sbase=10 MVA)
Qline_max_pu = 1.5

SOC_min_frac = 0.20
SOC_max_frac = 0.90
eta_ch  = 0.95
eta_dis = 0.95

# pu conversions (Sbase = 10,000 kW)
PV_cap_max_pu   = PV_cap_max_kW  / Sbase_kW
P_cap_max_pu    = BESS_p_max_kW  / Sbase_kW
E_cap_max_pu    = BESS_e_max_kWh / Sbase_kW   # kWh / kWbase = hours

# ─── Build model ──────────────────────────────
m = ConcreteModel()
m.N = Set(initialize=list(range(1, nb+1)))
m.L = Set(initialize=list(branches.keys()))
m.T = Set(initialize=T)

m.Pd    = Param(m.N, initialize={i: bus_load[i][0]/Sbase_kW   for i in range(1,nb+1)}, within=NonNegativeReals)
m.Qd    = Param(m.N, initialize={i: bus_load[i][1]/Sbase_kVAr for i in range(1,nb+1)}, within=NonNegativeReals)
m.lmult = Param(m.T, initialize={t: float(load_mult[t]) for t in T})
m.CF    = Param(m.T, initialize={t: float(pv_cf[t])    for t in T})
m.R     = Param(m.L, initialize={l: branches[l][2] for l in branches})
m.X     = Param(m.L, initialize={l: branches[l][3] for l in branches})
m.dt    = Param(initialize=dt)

# Planning binary + sizing vars
m.xPV = Var(m.N, within=Binary)
m.xB  = Var(m.N, within=Binary)
m.PPVmax  = Var(m.N, within=NonNegativeReals)
m.Pchmax  = Var(m.N, within=NonNegativeReals)
m.Pdismax = Var(m.N, within=NonNegativeReals)
m.Emax    = Var(m.N, within=NonNegativeReals)

# Operational vars (24 h)
m.Ppv   = Var(m.N, m.T, within=NonNegativeReals)
m.Pcurt = Var(m.N, m.T, within=NonNegativeReals)
m.Pch   = Var(m.N, m.T, within=NonNegativeReals)
m.Pdis  = Var(m.N, m.T, within=NonNegativeReals)
m.E     = Var(m.N, m.T, within=NonNegativeReals)
m.LS    = Var(m.N, m.T, within=NonNegativeReals)

# Network vars
m.Pflow = Var(m.L, m.T, within=Reals)
m.Qflow = Var(m.L, m.T, within=Reals)
m.V     = Var(m.N, m.T, within=Reals)

# Initial energy (linear aux — avoids Var×Var)
m.E_init = Var(m.N, within=NonNegativeReals)

print(f"Model built. Sbase={Sbase_kW} kW | Zbase={Zbase_ohm:.2f} Ω")
print(f"PV_cap_max = {PV_cap_max_pu:.3f} pu = {PV_cap_max_kW} kW per site")
print(f"BESS_e_max = {E_cap_max_pu:.3f} pu-h = {BESS_e_max_kWh} kWh per site")


In [ ]:
# ═══════════════════════════════════════════════
#  CONSTRAINTS  —  all linear (MILP-safe)
# ═══════════════════════════════════════════════
# NOTE ON SOC BOUNDS:
#   E[n,t] >= SOC_min * Emax[n]  is bilinear (Var×Var) → not LP/MILP-safe.
#   Fix: Emax[n] is interpreted as USABLE energy (the 20-90% window).
#   Physical rated capacity = Emax[n] / (SOC_max - SOC_min).
#   E[n,t] is then bounded in [0, Emax[n]] — both linear.
#   E_init[n] ∈ [0, Emax[n]] is also linear.

# --- Installation links (big-M, all linear) ---
m.c_pv_link = Constraint(m.N, rule=lambda m,n: m.PPVmax[n]  <= PV_cap_max_pu * m.xPV[n])
m.c_b1      = Constraint(m.N, rule=lambda m,n: m.Pchmax[n]  <= P_cap_max_pu  * m.xB[n])
m.c_b2      = Constraint(m.N, rule=lambda m,n: m.Pdismax[n] <= P_cap_max_pu  * m.xB[n])
m.c_b3      = Constraint(m.N, rule=lambda m,n: m.Emax[n]    <= E_cap_max_pu  * m.xB[n])

m.c_npv = Constraint(expr=sum(m.xPV[n] for n in m.N) <= Npv_max)
m.c_nb  = Constraint(expr=sum(m.xB[n]  for n in m.N) <= Nbess_max)

# --- PV: dispatched + curtailed = available CF × capacity ---
m.c_pv_avail = Constraint(m.N, m.T,
    rule=lambda m,n,t: m.Ppv[n,t] + m.Pcurt[n,t] == m.CF[t] * m.PPVmax[n])

# --- BESS power bounds ---
m.c_ch_cap  = Constraint(m.N, m.T, rule=lambda m,n,t: m.Pch[n,t]  <= m.Pchmax[n])
m.c_dis_cap = Constraint(m.N, m.T, rule=lambda m,n,t: m.Pdis[n,t] <= m.Pdismax[n])

# --- SOC dynamics ---
def soc_dyn(m, n, t):
    if t == 0:
        return m.E[n,t] == m.E_init[n] + (eta_ch*m.Pch[n,t] - (1/eta_dis)*m.Pdis[n,t])*m.dt
    return m.E[n,t] == m.E[n,t-1] + (eta_ch*m.Pch[n,t] - (1/eta_dis)*m.Pdis[n,t])*m.dt
m.c_soc = Constraint(m.N, m.T, rule=soc_dyn)

# SOC bounds: E in [0, Emax]  — LINEAR (Var <= Var is degree-1)
# Emax is USABLE capacity; rated = Emax / (SOC_max - SOC_min) = Emax / 0.70
m.c_soc_hi   = Constraint(m.N, m.T, rule=lambda m,n,t: m.E[n,t]     <= m.Emax[n])
m.c_einit_hi = Constraint(m.N,      rule=lambda m,n:   m.E_init[n]   <= m.Emax[n])
# (lower bounds of 0 are already enforced by NonNegativeReals)

# Daily cycle: end energy = initial energy  — LINEAR
m.c_cycle = Constraint(m.N, rule=lambda m,n: m.E[n,23] == m.E_init[n])

# --- Voltage ---
m.c_v_lo  = Constraint(m.N, m.T, rule=lambda m,n,t: m.V[n,t] >= Vmin)
m.c_v_hi  = Constraint(m.N, m.T, rule=lambda m,n,t: m.V[n,t] <= Vmax)
m.c_slack = Constraint(m.T,      rule=lambda m,t:   m.V[slack,t] == Vref)

# LinDistFlow: V_j = V_i - R*P - X*Q
def vdrop(m, l, t):
    i = from_bus[l]; j = to_bus[l]
    return m.V[j,t] == m.V[i,t] - m.R[l]*m.Pflow[l,t] - m.X[l]*m.Qflow[l,t]
m.c_vdrop = Constraint(m.L, m.T, rule=vdrop)

# Line flow limits
m.c_plim = Constraint(m.L, m.T, rule=lambda m,l,t: (-Pline_max_pu, m.Pflow[l,t], Pline_max_pu))
m.c_qlim = Constraint(m.L, m.T, rule=lambda m,l,t: (-Qline_max_pu, m.Qflow[l,t], Qline_max_pu))

# Active power balance
# Slack bus (bus 1): Pd=0, LS=0; balance holds trivially — skip to avoid
# over-constraining Pflow[branch_1] which is determined by downstream balance.
def p_balance(m, n, t):
    if n == slack:
        return Constraint.Skip          # substation supplies whatever P is needed
    Pin  = sum(m.Pflow[l,t] for l in in_lines[n])
    Pout = sum(m.Pflow[l,t] for l in out_lines[n])
    dem  = m.Pd[n] * m.lmult[t]
    return Pin - Pout == -(dem - m.LS[n,t] - m.Ppv[n,t] - m.Pdis[n,t] + m.Pch[n,t])
m.c_pbal = Constraint(m.N, m.T, rule=p_balance)

# Reactive power balance (PV at unity PF → Q_pv = 0)
# Slack bus (bus 1) is the grid connection — it provides unlimited Q; exclude it.
# All other buses: Q_in - Q_out = -Q_load
def q_balance(m, n, t):
    if n == slack:
        return Constraint.Skip          # grid supplies reactive power freely
    Qin  = sum(m.Qflow[l,t] for l in in_lines[n])
    Qout = sum(m.Qflow[l,t] for l in out_lines[n])
    return Qin - Qout == -m.Qd[n]*m.lmult[t]
m.c_qbal = Constraint(m.N, m.T, rule=q_balance)

# Load shedding bounded by demand
m.c_ls = Constraint(m.N, m.T, rule=lambda m,n,t: m.LS[n,t] <= m.Pd[n]*m.lmult[t])

# --- Objective: minimise ENS ---
m.obj = Objective(
    expr=sum(m.LS[n,t]*m.dt for n in m.N for t in m.T),
    sense=minimize
)

print("All constraints built — fully linear (MILP-safe).")
print("  Slack bus excluded from P/Q balance (grid = free source/sink).")
print("  Emax[n] = usable energy capacity (physical rated = Emax / 0.70)")


## 3. Solve

In [ ]:
def pick_solver():
    for name in ["cbc", "highs", "gurobi", "glpk"]:
        try:
            s = SolverFactory(name)
            if s.available():
                return name
        except Exception:
            pass
    return None

solver_name = pick_solver()
if solver_name is None:
    raise RuntimeError("No MILP solver found. Install: conda install -c conda-forge coincbc")
print("Using solver:", solver_name)

opt = SolverFactory(solver_name)
res = opt.solve(m, tee=True, load_solutions=False)

from pyomo.opt import TerminationCondition as TC
tc = res.solver.termination_condition
print("\nTermination condition:", tc)

if tc in [TC.optimal, TC.feasible]:
    m.solutions.load_from(res)
    print(f"✓ Solved.  ENS = {value(m.obj)*Sbase_kW:.2f} kWh/day")
else:
    print("\n✗ INFEASIBLE — running base-case voltage check...")
    # Quick diagnosis: compute base-case min voltage
    from collections import deque
    import numpy as np
    _ch={n:[] for n in range(1,nb+1)}; _pb={}
    for l,(fi,ti,r,x) in branches.items(): _ch[fi].append(ti); _pb[ti]=l
    _bfs=[1]; _q=deque([1])
    while _q:
        _node=_q.popleft()
        for c in _ch[_node]: _bfs.append(c); _q.append(c)
    _P={l:0.0 for l in branches}; _Q={l:0.0 for l in branches}
    for _node in reversed(_bfs[1:]):
        l=_pb[_node]; fi,ti,r,x=branches[l]
        _out=[ll for ll,(fii,tii,*_) in branches.items() if fii==_node]
        _P[l]=bus_load[_node][0]/Sbase_kW+sum(_P[ll] for ll in _out)
        _Q[l]=bus_load[_node][1]/Sbase_kVAr+sum(_Q[ll] for ll in _out)
    _V={1:1.0}
    for _node in _bfs[1:]:
        l=_pb[_node]; fi,ti,r,x=branches[l]
        _V[_node]=_V[fi]-r*_P[l]-x*_Q[l]
    minV=min(_V.values()); minbus=min(_V,key=_V.get)
    maxQ=max(_Q.values())
    print(f"  Base-case min voltage: {minV:.4f} pu at bus {minbus}  (Vmin limit={Vmin})")
    print(f"  Base-case max Q-flow:  {maxQ:.4f} pu  (Qline_max_pu={Qline_max_pu})")
    if minV < Vmin:
        print(f"  → Network violates Vmin={Vmin} BEFORE any DER is placed.")
        print(f"    The optimizer cannot fix this with ≤{Npv_max} PV sites.")
        print(f"    Try: set Vmin = {minV - 0.005:.3f} or lower in Cell 4.")
    if maxQ > Qline_max_pu:
        print(f"  → Max Q-flow {maxQ:.4f} exceeds Qline_max_pu={Qline_max_pu}.")
        print(f"    Try: set Qline_max_pu = {maxQ*1.1:.2f} or higher in Cell 4.")
    raise RuntimeError("Infeasible — adjust parameters in Cell 4 per diagnosis above.")


In [ ]:
# ═══════════════════════════════════════════════════════
#  OPTIONAL: Feasibility relaxation study
#  Run ONLY if the main MILP is infeasible.
#  Finds the tightest Vmin at which the base-case LP (no DER) is feasible.
# ═══════════════════════════════════════════════════════
from pyomo.environ import (
    ConcreteModel, Set, Param, Var, NonNegativeReals, Reals,
    Constraint, Objective, minimize, value
)
from pyomo.opt import SolverFactory, TerminationCondition as TC

def base_case_LP(Vmin_try, Vmax_try=1.05, Qlim=1.5, Plim=1.5, verbose=False):
    """LP: no DER, check if network P+Q balance + voltage is feasible.
    Slack bus excluded from P/Q balance (standard LinDistFlow convention).
    Uses load_solutions=False to avoid NoFeasibleSolutionError on new Pyomo.
    """
    mm = ConcreteModel()
    mm.N = Set(initialize=list(range(1, nb+1)))
    mm.L = Set(initialize=list(branches.keys()))
    mm.T = Set(initialize=T)
    mm.Pd    = Param(mm.N, initialize={i: bus_load[i][0]/Sbase_kW   for i in range(1,nb+1)}, within=NonNegativeReals)
    mm.Qd    = Param(mm.N, initialize={i: bus_load[i][1]/Sbase_kVAr for i in range(1,nb+1)}, within=NonNegativeReals)
    mm.lmult = Param(mm.T, initialize={t: float(load_mult[t]) for t in T})
    mm.R     = Param(mm.L, initialize={l: branches[l][2] for l in branches})
    mm.X     = Param(mm.L, initialize={l: branches[l][3] for l in branches})

    mm.Pflow = Var(mm.L, mm.T, within=Reals)
    mm.Qflow = Var(mm.L, mm.T, within=Reals)
    mm.V     = Var(mm.N, mm.T, within=Reals)
    mm.LS    = Var(mm.N, mm.T, within=NonNegativeReals)

    mm.c_v_lo  = Constraint(mm.N, mm.T, rule=lambda mm,n,t: mm.V[n,t] >= Vmin_try)
    mm.c_v_hi  = Constraint(mm.N, mm.T, rule=lambda mm,n,t: mm.V[n,t] <= Vmax_try)
    mm.c_slack = Constraint(mm.T, rule=lambda mm,t: mm.V[1,t] == 1.0)
    mm.c_plim  = Constraint(mm.L, mm.T, rule=lambda mm,l,t: (-Plim, mm.Pflow[l,t], Plim))
    mm.c_qlim  = Constraint(mm.L, mm.T, rule=lambda mm,l,t: (-Qlim, mm.Qflow[l,t], Qlim))

    def vdrop(mm, l, t):
        i = from_bus[l]; j = to_bus[l]
        return mm.V[j,t] == mm.V[i,t] - mm.R[l]*mm.Pflow[l,t] - mm.X[l]*mm.Qflow[l,t]
    mm.c_vdrop = Constraint(mm.L, mm.T, rule=vdrop)

    # Power balance — SKIP slack bus (bus 1); grid supplies freely
    def pb(mm, n, t):
        if n == 1: return Constraint.Skip
        Pin  = sum(mm.Pflow[l,t] for l in in_lines[n])
        Pout = sum(mm.Pflow[l,t] for l in out_lines[n])
        return Pin - Pout == -(mm.Pd[n]*mm.lmult[t] - mm.LS[n,t])
    mm.c_pbal = Constraint(mm.N, mm.T, rule=pb)

    def qb(mm, n, t):
        if n == 1: return Constraint.Skip
        Qin  = sum(mm.Qflow[l,t] for l in in_lines[n])
        Qout = sum(mm.Qflow[l,t] for l in out_lines[n])
        return Qin - Qout == -mm.Qd[n]*mm.lmult[t]
    mm.c_qbal = Constraint(mm.N, mm.T, rule=qb)

    mm.c_ls  = Constraint(mm.N, mm.T, rule=lambda mm,n,t: mm.LS[n,t] <= mm.Pd[n]*mm.lmult[t])
    mm.obj   = Objective(expr=sum(mm.LS[n,t] for n in mm.N for t in mm.T), sense=minimize)

    opt2 = SolverFactory(solver_name)
    # load_solutions=False avoids NoFeasibleSolutionError on new Pyomo
    res2 = opt2.solve(mm, tee=False, load_solutions=False)
    tc2  = res2.solver.termination_condition
    ok   = tc2 in [TC.optimal, TC.feasible]
    ens  = None
    if ok:
        mm.solutions.load_from(res2)
        ens = value(mm.obj)
    if verbose:
        print(f"  Vmin={Vmin_try:.2f} → {'FEASIBLE' if ok else 'INFEASIBLE'}"
              + (f", ENS={ens*Sbase_kW:.1f} kWh" if ens is not None else ""))
    return ok, ens

print("Scanning minimum feasible Vmin (base-case LP, no DER):")
for vmin_try in [0.95, 0.93, 0.90, 0.85, 0.80]:
    ok, ens = base_case_LP(vmin_try, verbose=True)
    if ok:
        print(f"  → Recommended: set Vmin ≤ {vmin_try:.2f} in Cell 4")
        break
else:
    print("  Network infeasible even at Vmin=0.80 — check data.")


## 4. Results — Siting, Sizing & Hosting Capacity

**Hosting Capacity (HC)** is the total installable DER capacity without violating voltage/thermal constraints:

$$HC_{PV} = \sum_{n \in \mathcal{N}_{PV}} P^{PV,max}_n \quad [\text{kW}]$$

$$HC_{BESS,E} = \sum_{n \in \mathcal{N}_{B}} E^{max}_n \quad [\text{kWh}]$$


In [ ]:
# ─── Siting ───────────────────────────────────
pv_sites   = [n for n in m.N if value(m.xPV[n]) > 0.5]
bess_sites = [n for n in m.N if value(m.xB[n])  > 0.5]

print("=== PV SITING ===")
print("PV buses:", pv_sites)
print("\n=== BESS SITING ===")
print("BESS buses:", bess_sites)

# ─── Sizing ───────────────────────────────────
print("\n=== PV SIZING ===")
HC_PV = 0.0
for n in pv_sites:
    kW = value(m.PPVmax[n]) * Sbase_kW
    HC_PV += kW
    print(f"  Bus {n:2d}: PPVmax = {kW:8.1f} kW")
print(f"  → Total HC_PV = {HC_PV:.1f} kW")

print("\n=== BESS SIZING ===")
HC_E = 0.0; HC_Pch = 0.0; HC_Pdis = 0.0
for n in bess_sites:
    Ekwh  = value(m.Emax[n])    * Sbase_kW
    Pch   = value(m.Pchmax[n])  * Sbase_kW
    Pdis  = value(m.Pdismax[n]) * Sbase_kW
    soc0  = value(m.E_init[n]) / max(value(m.Emax[n]), 1e-9)
    HC_E   += Ekwh; HC_Pch += Pch; HC_Pdis += Pdis
    E_rated_kWh = Ekwh / (SOC_max_frac - SOC_min_frac)  # usable / 0.70
    print(f"  Bus {n:2d}: Emax(usable)={Ekwh:7.1f} kWh  Erated={E_rated_kWh:7.1f} kWh | "
          f"Pch={Pch:6.1f} kW Pdis={Pdis:6.1f} kW")
print(f"  → Total HC_BESS_E    = {HC_E:.1f} kWh")
print(f"  → Total HC_BESS_Pch  = {HC_Pch:.1f} kW")
print(f"  → Total HC_BESS_Pdis = {HC_Pdis:.1f} kW")


## 5. Operational Checks — Voltage Profile & SOC

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 10

# ─── Voltage at each bus (worst hour = peak load = t=19) ──
t_peak = 19   # load_mult = 1.00
V_peak = np.array([value(m.V[n, t_peak]) for n in range(1, nb+1)])

violated_lo = np.where(V_peak < Vmin)[0] + 1
violated_hi = np.where(V_peak > Vmax)[0] + 1
print("Voltage check at peak-load hour (t=19):")
print(f"  Min V = {V_peak.min():.4f} pu | Max V = {V_peak.max():.4f} pu")
if len(violated_lo):
    print(f"  Under-voltage buses: {violated_lo.tolist()}")
else:
    print("  No under-voltage violations ✓")
if len(violated_hi):
    print(f"  Over-voltage buses:  {violated_hi.tolist()}")
else:
    print("  No over-voltage violations ✓")

fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(range(1, nb+1), V_peak, color='steelblue', alpha=0.75, width=0.8)
ax.axhline(Vmin, color='red',   ls='--', lw=1, label=f'Vmin={Vmin}')
ax.axhline(Vmax, color='green', ls='--', lw=1, label=f'Vmax={Vmax}')
ax.set_xlabel("Bus Number"); ax.set_ylabel("Voltage (p.u.)")
ax.set_title("Bus Voltage at Peak-Load Hour")
ax.legend(); plt.tight_layout(); plt.show()

# ─── SOC profile for BESS buses (all 24 h) ────
if bess_sites:
    fig, ax = plt.subplots(figsize=(7, 3))
    for n in bess_sites:
        # SOC_physical = SOC_min + (E/Emax) * (SOC_max - SOC_min)
        # Because E=0 → battery at 20% (not empty), E=Emax → battery at 90% (not full)
        E_vals   = [value(m.E[n,t])   for t in T]
        Emax_val =  value(m.Emax[n])
        soc = [SOC_min_frac + (e / Emax_val) * (SOC_max_frac - SOC_min_frac)
               for e in E_vals]
        ax.plot(T, soc, marker='o', ms=3, label=f'Bus {n}')
    ax.axhline(SOC_min_frac, color='red',   ls='--', lw=1, label='SOC_min=0.20')
    ax.axhline(SOC_max_frac, color='green', ls='--', lw=1, label='SOC_max=0.90')
    ax.set_xlabel("Hour"); ax.set_ylabel("SOC (p.u.)  [physical: 0.20–0.90]")
    ax.set_title("BESS State of Charge — Daily Cycle\n(E=0 → SOC=0.20, E=Emax → SOC=0.90)")
    ax.legend(); plt.tight_layout(); plt.show()


## 6. Hosting Capacity Summary Table & CSV Export

In [ ]:
rows = []
for n in m.N:
    rows.append({
        "Bus"         : n,
        "xPV"         : int(round(value(m.xPV[n]))),
        "PPVmax_kW"   : round(value(m.PPVmax[n]) * Sbase_kW, 2),
        "xBESS"       : int(round(value(m.xB[n]))),
        "Emax_kWh"    : round(value(m.Emax[n])    * Sbase_kW, 2),
        "Pchmax_kW"   : round(value(m.Pchmax[n])  * Sbase_kW, 2),
        "Pdismax_kW"  : round(value(m.Pdismax[n]) * Sbase_kW, 2),
    })
df = pd.DataFrame(rows)

# Pretty-print active rows
active = df[(df.xPV==1) | (df.xBESS==1)]
print(active.to_string(index=False))

df.to_csv("pv_bess_hosting_capacity_ieee33.csv", index=False)
print("\nSaved: pv_bess_hosting_capacity_ieee33.csv")

print(f"\n{'='*40}")
print(f" HC_PV   total : {HC_PV:.1f} kW")
print(f" HC_BESS total : {HC_E:.1f} kWh  |  {HC_Pch:.1f} kW charge  |  {HC_Pdis:.1f} kW discharge")
print(f" ENS            : {value(m.obj)*Sbase_kW:.2f} kWh/day")
print(f"{'='*40}")


## 7. Standalone Diagnostics (retained from original)
These helper cells work independently of the optimizer — useful for quick checks during debugging.

In [ ]:
# Voltage diagnostics — post-solve or for manual arrays
import numpy as np

def check_voltage(V_arr, Vmin=0.95, Vmax=1.05):
    """Check voltage array for violations."""
    vmin = V_arr.min(); vmax = V_arr.max()
    violated = np.where((V_arr < Vmin) | (V_arr > Vmax))[0]
    print(f"Minimum voltage : {vmin:.4f} pu")
    print(f"Maximum voltage : {vmax:.4f} pu")
    if len(violated):
        print(f"Violated buses  : {(violated+1).tolist()}")
    else:
        print("No voltage violations ✓")
    return violated

# Example
V_example = np.array([1.00, 0.98, 0.96, 0.94, 1.02])
check_voltage(V_example)


In [ ]:
# SOC diagnostics
import numpy as np

def check_soc(SOC_arr, SOCmin=0.20, SOCmax=0.90):
    """Check SOC array for violations."""
    print(f"SOC min = {SOC_arr.min():.3f}  |  SOC max = {SOC_arr.max():.3f}")
    viol = np.where((SOC_arr < SOCmin) | (SOC_arr > SOCmax))[0]
    if len(viol):
        print(f"Violations at t = {viol.tolist()}  ← would be infeasible in optimizer")
    else:
        print("No SOC violations ✓")
    return viol

SOC_example = np.array([0.50, 0.47, 0.45, 0.42, 0.44, 0.60])
check_soc(SOC_example)


In [ ]:
# SOC simulation — verify dynamics match optimizer convention
import numpy as np

def simulate_soc(P_bess, E_rated_kWh, SOC0=0.50, eta_c=0.95, eta_d=0.95, dt=1.0):
    """
    P_bess: array of kW (negative = charge, positive = discharge)
    Returns SOC array (length = len(P_bess)+1, starts at SOC0).
    Convention: consistent with MILP E[t] = E[t-1] + eta_c*Pch - (1/eta_d)*Pdis
    """
    SOC = [SOC0]
    for p in P_bess:
        if p < 0:   # charging
            dSOC = eta_c * (-p) * dt / E_rated_kWh
            SOC.append(SOC[-1] + dSOC)
        else:       # discharging
            dSOC = p * dt / (eta_d * E_rated_kWh)
            SOC.append(SOC[-1] - dSOC)
    return np.array(SOC)

P_example = np.array([-2, -2, 3, 3, -1])   # kW
soc_trace = simulate_soc(P_example, E_rated_kWh=10.0)
print("SOC trace:", np.round(soc_trace, 4))
print("Cycle balance (end - start):", round(soc_trace[-1] - soc_trace[0], 6),
      "← should be 0 for feasible daily cycle")
